## **Generating RAG Answers**

In the first part of this module, we evaluated search quality. We checked whether the right document appeared in the search results.

Now we evaluate the full RAG pipeline. For each generated question, we run RAG and save the answer produced by the LLM. Later, we'll compare this answer with the original FAQ answer.

This is the A->Q->A' setup:

A = original answer in the FAQ
Q = generated question from this answer
A' = answer produced by our RAG system
If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our questions came from FAQ records. For each question, we know which original answer it came from.

## Loading the data

Create a new notebook for RAG evaluation.

Load the ground truth questions:

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the FAQ documents and the search index:

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table for the original FAQ documents:

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

We'll use this lookup table to find the original answer for each ground truth question.

## Running RAG

Import the usual things first:

In [4]:
# from dotenv import load_dotenv
from openai import OpenAI

# load_dotenv()
openai_client = OpenAI()

For this lesson, use *RAGWithUsage* from the evaluation utilities. It subclasses *RAGBase* from module 01, so it has the same *rag* method.

It stores token usage after each LLM call. Then we can calculate the total cost later.

It also uses the search boosts we selected in the search tuning lesson: question=1.0, answer=2.0, and section=0.1.

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, *RAGBase* searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer. We save the answer so the next lesson can judge it.

Run RAG for one question:

In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join late. If you want a certificate, make sure to submit your project while submissions are still open.'

Check the cost of this call:

In [7]:
assistant.total_cost()

0.0004785

Get the original answer from the document ID:

In [8]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Now save both answers in one record:

In [9]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join late. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

## Processing all questions

Create a function that processes one ground truth record:

In [10]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

Test it on one record:

In [11]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join late. If you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Before running the full batch, reset the usage we collected while testing:

In [12]:
assistant.reset_usage()

This calls the LLM once per ground truth question, so it can take some time. Let's process the questions in parallel and track progress.

Import the parallel processing helper from the same utility file:

In [13]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

Run RAG for all ground truth questions:

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

*generate_rag_answer* returns one answer record for each question.

Collect the answer records:

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

Calculate the total cost:

In [ ]:
assistant.total_cost()

Save the answers:

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

We generated this file for the course materials on May 29, 2026. The run used 395 ground truth questions.

The total cost was $0.34332825, about 34 cents.

If you don't want to generate the RAG answers yourself, download the file we prepared:

In [ ]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

# wget -O data/rag-answers-new.csv ${PREFIX}/04-evaluation/data/rag-answers-new.csv

In the next lesson, we'll evaluate these answers with an LLM judge.

## **LLM as a Judge**

In the previous lesson, we generated RAG answers for our ground truth questions. Now we need to decide whether these answers are good enough.

For offline evaluation, we have three things:

* the original FAQ answer
* the question generated from that answer
* the answer generated by our RAG pipeline

An LLM judge is another LLM call that compares these three pieces. We ask it whether the RAG answer recovers the same information as the original answer.

It can also explain why an answer is bad.

For example:

* the retrieved document might be wrong
* the answer might miss the key point
* the model might say that it doesn't know

This approach is useful when exact text matching is too strict. The RAG answer doesn't need to copy the FAQ answer word for word. It needs to answer the question with the same key information.

This evaluates the full RAG flow in one pass:

* search: did we retrieve context that contains the answer?
* prompt: did we give the model enough context to answer?
* LLM: did the model produce a useful answer from that context?

If the judge marks an answer as bad, we still need to look at the example. The judge tells us where to investigate. It doesn't replace reading the failing cases.

## Loading the RAG answers

Start from the CSV we created in the previous lesson:

In [19]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

Each row has the generated question, the original FAQ answer, and the answer produced by the RAG pipeline.

This is offline evaluation. We can do it because our test dataset came from FAQ records. We know the original answer for every generated question.

In production, we usually don't have that original answer for real user questions. There we can still use an LLM judge. The prompt has to judge only the question and the generated answer. In this lesson, we use the stronger offline setup.

## A->Q->A' evaluation

We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.

First, define the output format:

In [20]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

The judge returns two fields. The *score* gives us a metric we can aggregate. The *reasoning* explains the score, which helps when we look at bad examples.

First, write the judge instructions. This tells the judge what to compare and how to assign the score.

In [21]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

Then define the prompt template. This is the data we pass to the judge for each answer.

In [22]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

Import the structured-output helper:

In [23]:
# from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

# load_dotenv()
openai_client = OpenAI()

Take one record:

In [24]:
rec = answers[0]

Create the judge prompt:

In [25]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

Call the judge:

In [26]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project while submissions are still open. It is semantically equivalent.', score='good')

Check the cost:

In [27]:
calc_price(usage)

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.0002385,
 'total_cost': 0.00046125}

Now put the same logic into a function:

In [28]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

Test it on the same record:

In [29]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, but certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

## Running the judge

Run the evaluation on all answers:

In [30]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

Use the same parallel processing helper:

In [31]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/395 [00:00<?, ?it/s]

Split the results:

In [32]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [33]:
df_eval = pd.DataFrame(evaluations)

Calculate the total cost:

In [34]:
calc_total_price(usages)

0.2531489999999999

Check the results:

In [35]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 380/395 = 96.20%


Look at the "bad" cases to understand what went wrong:

In [36]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
15,How do the free GPU hours work on these cloud ...,c6c2888275,bad,The AI answer does not address the question or...
29,Is peer-review of the capstone project require...,69d122f12e,bad,The ground truth says peer-review is tied to t...
38,How will I know when a module is actually read...,96286b4be4,bad,The AI answer does not convey the ground truth...
106,Do I need an OpenAI API key just to check how ...,fe8fed31e6,bad,The AI answer does not answer the question. Th...
124,What model or provider should I switch to if I...,6e1d8a7b29,bad,The AI answer does not answer the question or ...


These rows are often the most useful part of the evaluation. They can show that search retrieved the wrong document. They can also show that the answer is too generic. Sometimes the RAG pipeline says that it doesn't know even though the FAQ had the answer.

## Evaluating the judge

The judge can be wrong. It may rate an answer as good even though search failed to retrieve the right document. In that case the judge is too lenient. Make the instructions stricter and re-run the evaluation.

To evaluate the judge, you need to look at the results yourself. Sample some good and bad cases, read the judge reasoning, and check whether you agree with the verdict. You cannot use another judge to evaluate the judge. This is manual work, but it is necessary.

A practical approach is to build a simple application with Streamlit. Show each question, the original answer, the generated answer, and the judge verdict side by side. Then mark each verdict as correct or incorrect and use that feedback to adjust the judge instructions. This is a lot of trial and error, but it makes the evaluation framework more reliable.

## Saving the results

Save the judged answers:

In [37]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

We generated this file for the course materials on May 29, 2026. The run used 395 RAG answers.

The results were:

* Good: 379
* Bad: 16
* The total cost was $0.251331, about 25 cents.

If you don't want to run the judge yourself, download the file we prepared:

In [ ]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget -O data/rag-evaluations-new.csv ${PREFIX}/04-evaluation/data/rag-evaluations-new.csv

We now have an answer-quality score for the RAG pipeline. In the next lesson, we'll apply the same idea to an agent and also capture the tool calls it made.